# 🚀 Laya Decision Engine - Kaggle Server (CPU Only + Multilingual)

Notebook tối giản chạy **Laya Server** trên **CPU Kaggle** (không cần bật GPU, tiết kiệm hạn ngạch GPU quota 30h/tuần) và public qua **ngrok** với static domain.
Cài đặt trực tiếp từ fork repository [binhminhanh1235/laya-s1-decision](https://github.com/binhminhanh1235/laya-s1-decision) (đã tích hợp các bản vá lỗi và kiểm soát bộ nhớ ).

### ⚠️ Yêu cầu trước khi chạy:
1. **Bật Internet**: Tại thanh bên phải Notebook  ->  -> chọn **Internet On**.
2. **Thêm Secret**: Menu trên cùng ->  ->  -> thêm secret với Label  và Value là auth token Ngrok của bạn.
3. **Accelerator**: Có thể để  (CPU) để không tốn GPU quota của Kaggle.

In [ ]:
# =============================================================================
# 🚀 1 CELL DUY NHẤT: CÀI ĐẶT, KẾT NỐI NGROK & CHẠY LAYA SERVER TRÊN CPU
# =============================================================================
import os
from kaggle_secrets import UserSecretsClient

# 1. Cài đặt pyngrok và laya kèm serve & mcp
!pip install -q pyngrok "git+https://github.com/binhminhanh1235/laya-s1-decision.git#egg=laya[serve,mcp]"

from pyngrok import ngrok

# 2. Lấy Ngrok Token từ Kaggle Secrets & cấu hình
user_secrets = UserSecretsClient()
ngrok_token = user_secrets.get_secret("NGROK_AUTHTOKEN")
ngrok.set_auth_token(ngrok_token)

# Đóng tunnel cũ (nếu có khi rerun lại cell) để tránh xung đột domain tĩnh
ngrok.kill()

# 3. Mở tunnel ngrok với domain cố định
public_url = ngrok.connect(8000, domain="ferret-above-formerly.ngrok-free.app")
print(f"\n{'='*55}")
print(f"👉 Public Domain:    {public_url}")
print(f"👉 Health Check:     {public_url}/health")
print(f"👉 Inference API:    {public_url}/v1/systemone")
print(f"👉 Device:           CPU (Kaggle {os.cpu_count() or 4} cores)")
print(f"👉 Active Checkpoint: multilingual only")
print(f"{'='*55}\n")

# 4. CẤU HÌNH ÉP CHẠY TRÊN CPU & CHỈ DÙNG CHECKPOINT MULTILINGUAL
os.environ["CUDA_VISIBLE_DEVICES"] = ""             # Vô hiệu hóa GPU, không cấp phát VRAM
os.environ["LAYA_DEVICE"] = "cpu"                   # Bắt buộc Laya chạy inference trên CPU
os.environ["LAYA_MODELS"] = "multilingual"          # Chỉ tải duy nhất multilingual
os.environ["LAYA_MAX_LOADED"] = "1"                # Giới hạn giữ tối đa 1 model trong RAM/VRAM
os.environ["LAYA_THREADS"] = str(os.cpu_count() or 4) # Tối ưu đa luồng CPU (intra-op threads)

# 5. Khởi động Laya server trên port 8000
!python -m laya.serve